# Volatility Dynamics 14 — Seven horizons, four distribution models and an augmented Markov lift

**Status: preregistered, historical exploratory, not executed.** The historical 2024–2026 source has been inspected in prior modules; this does **not** produce untouched confirmation.

**Seven horizons:** 5, 10, 15, 30, 60, 120, 240 minutes. **Five models:** M1 Gaussian two-EMA, M2 Student-t, M3 constant broad-tail Gaussian mixture, M4 phase-conditioned broad-tail mixture, and AM two-state-proxy Markovian lift with propagated variance and five-minute shock mechanism.

Read `discussions/Volatility_Dynamics/031_chatgpt_vd14_markov_lift_preregistration.md` before interpreting the experiment. The rare Gaussian mixture component is **not a labelled physical discontinuity**. The augmented Markov model uses observable EMA proxies (not an identified hidden state or exact jump diffusion).


## 0. Environment, parameters and synthetic tests

In [ ]:
from pathlib import Path
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').exists():
    ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').exists(), 'Run the notebook within deribit-market-data'
load_dotenv(ROOT/'.env');load_dotenv()
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from data_engineering.modeling import markov_lift_distributions as ml
print('VD14',ml.VERSION)
print(json.dumps(ml.registration(),indent=2))
OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'14'
OUT.mkdir(parents=True,exist_ok=True)
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
ASSETS=tuple(a.strip().upper() for a in os.getenv('VD14_ASSETS','AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if a.strip())
BOOT=int(os.getenv('VD14_BOOT',str(ml.BOOT)))
assert BOOT>=100, 'For diagnostic runs set VD14_BOOT>=100; normal final run 1000'
print('History:',HIST,'| assets:',ASSETS,'| bootstrap:',BOOT)
START=time.monotonic()


## 1. Raw five-minute spot closes, independent causal features and targets

In [ ]:
raw,excluded=ml.load_returns(HIST,ASSETS)
print('Exclusions:',excluded,'| raw rows:',len(raw))
panel=ml.panelize(raw)
del raw
features=ml.make_features(panel)
frame=ml.make_targets(panel,features)
print('Prepared in',round(time.monotonic()-START,1),'s')
print(panel.groupby('split',sort=False).agg(days=('date','nunique'),rows=('return_5m','size'),valid_returns=('return_5m','count')).to_string())
print('Eligible origins before target censoring:',frame.groupby('split').ready.sum().to_dict())
print('First and last bars:',str(panel.state_time.min()),'->',str(panel.state_time.max()))
assert tuple(ml.HORIZONS)==(5,10,15,30,60,120,240)
assert set(frame.split.unique())=={'train','valid','test'}


## 2. Fit five distributions at each horizon and bootstrap in UTC-day blocks

The one-step AM shock mechanism and bivariate state transition are fitted from train data (with validation selection of the 5m wide-component scale). Longer-horizon AM forecasts propagate these fitted states. M1–M4 fit separately at each horizon. **Origin-level predictions are discarded after scoring each horizon**, keeping output small and memory bounded.


In [ ]:
scores,boot,parameters=ml.run_all(frame,B=BOOT)
print('Models:',sorted(scores.model.unique().tolist()))
print('Horizon coverage:')
print(scores.groupby('horizon_min').agg(n_models=('model','nunique'),origins=('n_origins','min'),days=('n_days','min')).to_string())
assert len(scores)==len(ml.MODELS)*len(ml.HORIZONS)
assert (scores.groupby('horizon_min').n_origins.nunique()==1).all()
print('Scored in',round(time.monotonic()-START,1),'s')


## 3. All model metrics; 60-minute primary in detail

In [ ]:
cols=['horizon_min','model','nll','rv_log_mse','tail_logloss','tail_brier','tail_observed','tail_predicted','interval_miss95']
display(scores[cols].round(6))
display(boot[(boot.horizon_min==ml.PRIMARY)&(boot.metric.isin(['nll','tail_logloss']))].round(6))
# Calibration is not optional: all five models predict physical absolute-return tails.
print('Two-sided nominal interval miss rate = 0.05; observed values outside 0.05 indicate calibration differences.')


## 4. Predeclared summary (not a tradability or prospective-confirmation gate)

In [ ]:
result=ml.decision(scores,boot)
print(json.dumps(result,indent=2))
assert result['status'] in ('HISTORICAL_EXPLORATORY','INSUFFICIENT_DATA')
assert not result['not_prospective'] is False


## 5. Save only compact reproducible artifacts

In [ ]:
scores.to_csv(OUT/'14_scores.csv',index=False,float_format='%.9g')
boot.to_csv(OUT/'14_day_bootstrap.csv',index=False,float_format='%.9g')
(OUT/'14_decision.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
manifest={'module':'Volatility_Dynamics_14','version':ml.VERSION,
  'registration':ml.registration(),'source_dir':str(HIST),
  'assets':sorted(panel.underlying.unique()),'excluded_assets':excluded,
  'earliest_date':str(panel.state_time.min()),'latest_date':str(panel.state_time.max()),
  'day_splits':{str(k):int(v) for k,v in panel.groupby('split').date.nunique().items()},
  'bootstrap_draws':BOOT,'models':list(ml.MODELS),
  'parameter_fits':parameters,'decision':result,
  'scientific_caveat':'Observational Markov-proxy transitions and statistical tail mixtures; previous data exposure prevents prospective inference.'}
(OUT/'14_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Saved 4 compact artifacts under',OUT)
print('No raw origin predictions, return tensors or market data were saved. No database writes.')
